In [23]:
import pandas as pd

In [26]:
df = pd.read_parquet("../data/candidate_detail.parquet")

In [27]:
pd.set_option("display.max_rows", None)

In [ ]:
c = pd.read_parquet("../data/candidate_detail.parquet")
c = c[c.outcome.isin(["Approved","Commercialized","Failed Phase 1","Failed Phase 2","Failed Phase 3"])].copy()
c["y"]     = c.outcome.isin(["Approved","Commercialized"]).astype(int)
c["drug"]  = c.candidate_id.str.split("__").str[0]          # drug identity (memorization key)
c["year"]  = pd.to_datetime(c.earliest_start_date, errors="coerce").dt.year
c["split"] = (c.year <= 2019).map({True:"train", False:"test"})
# join the model's test predictions to inspect seen/new-drug behavior:
#pred = pd.read_parquet("../runs/xgb_di_2019/predictions.parquet")  # candidate_id, y_true, y_proba
#df = pred.merge(c[["candidate_id","drug","outcome","year"]], on="candidate_id", how="left")
#df["seen_drug"] = df.drug.isin(set(c[c.split=="train"].drug))

KeyError: 'candidate_id'

In [20]:
df.seen_drug.value_counts()

seen_drug
True     6458
False     564
Name: count, dtype: int64

In [21]:
# ROC AUC on seen vs. new drugs:
from sklearn.metrics import roc_auc_score
print("ROC AUC on seen drugs:", roc_auc_score(df[df.seen_drug].y_true, df[df.seen_drug].y_proba))
print("ROC AUC on new drugs:", roc_auc_score(df[~df.seen_drug].y_true, df[~df.seen_drug].y_proba))

ROC AUC on seen drugs: 0.8183791551584638
ROC AUC on new drugs: 0.6255754915881416


In [22]:
# PR AUC on seen vs. new drugs:
from sklearn.metrics import average_precision_score
print("PR AUC on seen drugs:", average_precision_score(df[df.seen_drug].y_true, df[df.seen_drug].y_proba))
print("PR AUC on new drugs:", average_precision_score(df[~df.seen_drug].y_true, df[~df.seen_drug].y_proba))

PR AUC on seen drugs: 0.683141583201042
PR AUC on new drugs: 0.4760190248390718
